In [ ]:
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "jupytergis==0.16.8",
#     "openeo",
# ]
# ///

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run notebooks/bais2_burned_area_sentinel2.ipynb
```

(The `uvx` command is included when installing `uv`).

## License

This notebook is an adaptation of the openEO-UDP notebook
[bais2_burned_area.ipynb](https://github.com/developmentseed/openeo-udp/blob/main/notebooks/sentinel/sentinel-2/fire_and_disaster_monitoring/bais2_burned_area.ipynb)
by Development Seed. That notebook is a conversion of a Sentinel Hub evalscript and is licensed under **CC-BY-SA-4.0**.
This adaptation is also licensed under **CC-BY-SA-4.0**.

- Original evalscript: https://custom-scripts.sentinel-hub.com/sentinel-2/bais2/ (Harel Dan)
- Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
- Conversion: Development Seed (openEO-UDP project)

# Burned area of the 2025 Portugal wildfires with BAIS2, September 2025

This notebook maps the burn scars of the summer 2025 wildfires in the Beiras e Serra da Estrela region, Portugal, with the Burned Area Index for Sentinel-2 (BAIS2), from Sentinel-2 L2A data with openEO.
It shows the result as a dynamic tile layer in a JupyterGIS map, on top of an OpenStreetMap basemap.

The same example is available in the [EOPF openEO Studio](https://studio.explorer.eopf.copernicus.eu/editor/sentinel-2-bais2), which uses the EOPF Sentinel-2 Zarr products.

BAIS2 uses the red-edge bands of Sentinel-2, which are sensitive to the chlorophyll content of vegetation, with the NIR and SWIR bands:

$$
BAIS2 = \Bigl(1 - \sqrt{\frac{B06 \times B07 \times B8A}{B04}}\Bigr) \times \Bigl(\frac{B12 - B8A}{\sqrt{B12 + B8A}} + 1\Bigr)
$$

The steps are:
1. Connect to the openEO backend.
2. Load the Sentinel-2 L2A scene of 25 September 2025 and make a first-valid-pixel mosaic.
3. Calculate BAIS2 and color it from blue (low burn severity) through yellow to red (high burn severity).
4. Show the result in JupyterGIS.

## Connect to the openEO backend

This notebook uses the openEO backend at `https://openeo.ds.io/`. You log in with a [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/) account.

In [ ]:
import openeo
from openeo.processes import array_create, if_

connection = openeo.connect("https://openeo.ds.io/")

In [ ]:
connection.authenticate_oidc()

## Parameters

The bounding box covers the mountains of the Beiras e Serra da Estrela region, in central Portugal, where wildfires burned in the summer of 2025.

The time range is 25 September 2025, after the summer fire season, on a cloud-free day. The end date is exclusive in openEO.
The cloud cover filter keeps only scenes with 20 % cloud cover or less.

In [ ]:
location_name = "Beiras e Serra da Estrela, Portugal"

bounding_box = {"west": -7.8, "south": 40.1, "east": -7.0, "north": 40.9}
temporal_extent = ["2025-09-25", "2025-09-26"]
max_cloud_cover = 20

# Sentinel-2 bands for the algorithm, in this order:
# B04 (red), B06 (red edge 2), B07 (red edge 3), B8A (narrow NIR), B12 (SWIR 2190 nm)
bands = ["B04_20m", "B06_20m", "B07_20m", "B8A_20m", "B12_20m"]

## Load the Sentinel-2 data

BAIS2 is designed for 20 m resolution, so we use the 20 m version of each band.
The backend applies the scale and offset of the Sentinel-2 L2A products, so the bands are reflectance values from 0 to 1 and are used without scaling.

- **B04**: Red, 664.6 nm
- **B06**: Red edge, 740.5 nm
- **B07**: Red edge, 782.8 nm
- **B8A**: Narrow NIR, 864.7 nm
- **B12**: SWIR, 2202.4 nm

In [ ]:
s2cube = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=bounding_box,
    temporal_extent=temporal_extent,
    bands=bands,
    properties={"eo:cloud_cover": lambda x: x <= max_cloud_cover},
)

s2cube = s2cube.process(
    "apply_pixel_selection",
    pixel_selection="first",
    data=s2cube,
)

## BAIS2 burned area visualization

The BAIS2 value is shown with a color ramp:

| BAIS2 | Color | Meaning |
|---|---|---|
| < 0 | White | No burn |
| 0 to 0.5 | Blue to yellow | Low to moderate burn severity |
| 0.5 to 0.99 | Yellow to red | Moderate to high burn severity |
| > 0.99 | Magenta | Very high values, such as active fires |

In [ ]:
def bais2_visualization(data):
    """
    Map burned areas with the Burned Area Index for Sentinel-2 (BAIS2).
    Input data array: [B04, B06, B07, B8A, B12]

    BAIS2 = (1 - sqrt((B06 * B07 * B8A) / B04)) * ((B12 - B8A) / sqrt(B12 + B8A) + 1)
    """
    B04, B06, B07, B8A, B12 = data[0], data[1], data[2], data[3], data[4]

    bais = (1 - ((B06 * B07 * B8A) / B04) ** 0.5) * (
        (B12 - B8A) / ((B12 + B8A) ** 0.5) + 1
    )

    min_val = 0.0
    max_val = 0.99
    zero_threshold = 0.5  # break point of the color ramp

    # Spatial ones to build constant colors with the shape of the image
    spatial_ones = bais * 0 + 1

    def color(r, g, b):
        return array_create([spatial_ones * r, spatial_ones * g, spatial_ones * b])

    underflow = color(1.0, 1.0, 1.0)  # white: no burn
    low_color = color(0.0, 0.0, 1.0)  # blue: low burn severity
    zero_color = color(250 / 255, 255 / 255, 10 / 255)  # yellow: moderate
    high_color = color(255 / 255, 20 / 255, 20 / 255)  # red: high burn severity
    overflow = color(1.0, 0.0, 1.0)  # magenta: active fire

    def linear_interpolation(color_a, color_b, bais_val, val_min, val_max):
        t = (bais_val - val_min) / (val_max - val_min)
        r = color_a[0] + (color_b[0] - color_a[0]) * t
        g = color_a[1] + (color_b[1] - color_a[1]) * t
        b = color_a[2] + (color_b[2] - color_a[2]) * t
        return array_create([r, g, b])

    return if_(
        bais < min_val,
        underflow,
        if_(
            bais < zero_threshold,
            linear_interpolation(low_color, zero_color, bais, min_val, zero_threshold),
            if_(
                bais <= max_val,
                linear_interpolation(
                    zero_color, high_color, bais, zero_threshold, max_val
                ),
                overflow,
            ),
        ),
    )


# Apply BAIS2 on the data cube on the bands dimension
bais2_image = s2cube.apply_dimension(
    dimension=s2cube.metadata.band_dimension.name, process=bais2_visualization
)

# Linear scale to 0-255 for RGB visualization
bais2_image = bais2_image.linear_scale_range(
    input_min=0, input_max=1, output_min=0, output_max=255
)

## False color image

A false color (color infrared) image of the same scene gives context under the BAIS2 layer.
It puts the narrow NIR band (B8A) in the red channel, red (B04) in green and green (B03) in blue, with the same filters and pixel selection.
Healthy vegetation reflects NIR strongly, so it is bright red. Burn scars have lost their vegetation and are dark brown to black, so they stand out clearly.
B8A is a 20 m band, while B04 and B03 are 10 m bands.
The bands are already 0–1 reflectance, so we scale 0–0.3 to 0–255 to make the image brighter.

In [ ]:
false_color = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=bounding_box,
    temporal_extent=temporal_extent,
    bands=["B8A_20m", "B04_10m", "B03_10m"],
    properties={"eo:cloud_cover": lambda x: x <= max_cloud_cover},
)
false_color = false_color.process(
    "apply_pixel_selection",
    pixel_selection="first",
    data=false_color,
)
false_color = false_color.linear_scale_range(
    input_min=0, input_max=0.3, output_min=0, output_max=255
)

## Show the result in JupyterGIS

The map has an OpenStreetMap basemap. Two openEO layers are added on top of it: the false color image, then the BAIS2 layer.
Hide the BAIS2 layer in the layer panel to see the false color image.
Each `add_openeo_tile_layer` call creates an XYZ service on the backend and fits the map to the bounding box.

The backend calculates each tile on request, so the layer fills in slowly.

In the BAIS2 layer, yellow to red areas are burn scars and white is land without burn.
In the false color layer, burn scars are dark brown to black and vegetation is red.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=40.5, longitude=-7.4, zoom=10)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors",
)

In [ ]:
doc.add_openeo_tile_layer(false_color, name="False Color")
doc.add_openeo_tile_layer(bais2_image, name="BAIS2 Burned Area")
doc

## Technical notes

**Applications:**
- Mapping and delineating burn scars after a wildfire.
- Assessing burn severity, and comparing it before and after a fire (dBAIS2).

**Limitations:**
- Water and cloud shadows are dark and can look like burn scars. The notebook does not mask them.
- The thresholds were calibrated for Mediterranean regions and can change with the fire intensity and the region.
- Removal of charcoal and early regrowth of vegetation can change the SWIR signal of older burn scars.
- A single date cannot separate burn scars from other dark surfaces as well as a difference of two dates (before and after the fire).

## Attribution

**Original script:** [BAIS2 (Burned Area Index for Sentinel-2)](https://custom-scripts.sentinel-hub.com/sentinel-2/bais2/) by Harel Dan, [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)

**Citation:** Filipponi, F. (2018). BAIS2 – Burned Area Index for Sentinel-2. *Proceedings*, 2(7), 364. https://doi.org/10.3390/ecrs-2-05177